# Agrupamento de Coordenadas

### Objetivo

Reduzir a granularidade espacial da base original de ocorrências de furtos e roubos da RMSP, agrupando ocorrências próximas em regiões comuns. Com isso, o modelo consegue capturar **padrões criminais por região**, em vez de tratar cada ponto isolado como uma unidade independente.

### Abordagem: grade hexagonal H3

Cada par de coordenadas (latitude, longitude) é convertido no código da célula H3 em que cai. Ocorrências na mesma célula passam a compartilhar o mesmo identificador, e a análise segue por agregações sobre essa coluna.

Vantagens da escolha:

- **Identificadores estáveis**: a mesma célula vale para treino, validação, teste e dados futuros.
- **Sem vazamento de dados**: a conversão é determinística e não aprende nada com a base.
- **Vizinhança uniforme**: os 6 vizinhos de um hexágono ficam à mesma distância, o que facilita features do entorno.

### Decisão: a resolução H3 será definida antes do treinamento

A resolução **não** será tratada como hiperparâmetro dos modelos. Uma busca de hiperparâmetros deve alterar apenas o modelo, e não a natureza dos dados, que é exatamente o que a resolução faz: ao mudá-la, mudam a unidade de análise, o alvo e a dificuldade do problema. Por isso, as métricas de resoluções diferentes não são diretamente comparáveis.

In [1]:
import h3
import pandas as pd

In [3]:
df_raw = pd.read_csv('datasets/database_processed.csv')

Como o objetivo é estimar o risco de furtos e roubos por região ou bairro, serão avaliadas as resoluções H3 **6, 7 e 8**, e a escolha final virá da densidade dos dados e da persistência temporal.

In [13]:
def diagnostico(df, res):
    d = df.copy()
    d["h3"] = [h3.latlng_to_cell(la, lo, res) for la, lo in zip(d["LATITUDE"], d["LONGITUDE"])]

    por_celula = d.groupby("h3").size()
    por_cel_mes = d.groupby(["h3", "ANO_BO", "MES_BO"]).size()

    n_cel = por_celula.size
    n_meses = d[["ANO_BO", "MES_BO"]].drop_duplicates().shape[0]
    preenchimento = len(por_cel_mes) / (n_cel * n_meses)

    return {
        "res": res,
        "celulas": n_cel,
        "mediana_furtos_por_celula": por_celula.median(),
        "media_furtos_por_celula_mes": por_cel_mes.mean() * round(100 * preenchimento, 1) / 100,
        "pct_celula_mes_nao_vazia": round(100 * preenchimento, 1),
    }

In [14]:
results_resolutions = pd.DataFrame([diagnostico(df_raw, r) for r in [4, 5, 6, 7, 8, 9]])

In [15]:
results_resolutions

,res,celulas,mediana_furtos_por_celula,media_furtos_por_celula_mes,pct_celula_mes_nao_vazia
0,4,46,1.0,721.541129,22.5
1,5,93,2.0,356.604282,39.0
2,6,272,102.0,121.918447,60.1
3,7,1222,58.0,27.118977,56.7
4,8,5628,34.5,5.885621,51.8
5,9,24412,29.0,1.356791,45.3


In [16]:
def persistencia(df, res):
    d = df.copy()
    d["h3"] = [h3.latlng_to_cell(la, lo, res) for la, lo in zip(d["LATITUDE"], d["LONGITUDE"])]
    anual = d.groupby(["h3", "ANO_BO"]).size().unstack(fill_value=0)
    return anual.corr(method="spearman")

In [18]:
persistencia(df_raw, 7)

ANO_BO,2022,2023,2024,2025,2026
ANO_BO,,,,,
2022,1.000000,0.964966,0.960780,0.954382,0.939419
2023,0.964966,1.000000,0.960381,0.951444,0.938275
2024,0.960780,0.960381,1.000000,0.960629,0.944603
2025,0.954382,0.951444,0.960629,1.000000,0.944981
2026,0.939419,0.938275,0.944603,0.944981,1.000000


In [19]:
persistencia(df_raw, 8)

ANO_BO,2022,2023,2024,2025,2026
ANO_BO,,,,,
2022,1.000000,0.942736,0.937646,0.928109,0.913972
2023,0.942736,1.000000,0.943748,0.932465,0.918155
2024,0.937646,0.943748,1.000000,0.942264,0.921474
2025,0.928109,0.932465,0.942264,1.000000,0.928282
2026,0.913972,0.918155,0.921474,0.928282,1.000000
